# 1H Structure + 15m Execution — LVN Backtest (v7)

**Dual-timeframe architecture:**
- **1H**: Auction structure — session-anchored volume profile, LVN detection, developing POC/VAH/VAL, balance regime filter, cross-session zone persistence.
- **15m**: Execution — rejection detection, entry timing, stop placement, TP/SL fills.

**Why:** Rejection events often happen inside a 1H candle and are invisible at 1H resolution. By executing on 15m while using 1H for structure, we see rejections that the 1H-only backtest missed.

## Cell 1 — Imports & Configuration

In [23]:
import math
import os
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Optional, Tuple
from collections import deque

import numpy as np
import pandas as pd

RUN_VERSION = "v7_1H_structure_15m_execution"

CONFIG = {
    # Data
    "symbol": "BTCUSDT",
    "path_1h": "data/btc_1h.csv",
    "path_15m": "data/btc_15m_perps.csv",

    # Session anchoring (applied to 1H)
    "anchor_type": "Daily",
    "fixed_bars": 100,

    # Volume profile (1H)
    "num_rows": 24,
    "value_area_pct": 70,

    # LVN detection (1H)
    "lvn_percentile": 30,
    "min_lvn_height_pct": 10.0,  # percent of profile rows (10.0 => 10%)
    "min_session_maturity_pct": 30,
    "max_lvn_boxes": 10,

    # Zone persistence
    "zone_max_age_sessions": 3,

    # Entry (15m execution)
    "entry_mode": "zone_edge",
    "inner_offset_bps": 2,
    "order_ttl_bars": 12,               # TTL in 15m bars (= 3 hours)
    "one_order_at_a_time": True,
    "require_rejection": False,
    "min_rejection_wick_ratio": 0.0,

    # Stops & targets (15m)
    "tp_target": "poc",
    "sl_buffer_bps": 10,
    "max_hold_bars_15m": 24,             # time stop in 15m bars (= 6 hours)
    "min_edge_bps": 30,

    # Fill model
    "conservative_same_bar_tp": True,
    "fee_bps": 0,
    "slip_bps": 0,

    # Filters (1H structure)
    "use_poc_magnet_filter": False,
    "use_va_containment_filter": True,
    "va_containment_lookback": 24,       # 24 x 1H = 24 hours
    "va_containment_threshold": 0.60,

    # Debug toggles
    "debug_disable_va_filter": False,
    "debug_disable_rejection": False,
    "debug_disable_magnet_filter": False,
    "debug_min_edge_bps_override": None,
    "debug_h1_trace_bars": 0,
    "debug_lvn_diag_sessions": 0,
}


print(f"Config loaded ({RUN_VERSION})")
print("debug_lvn_diag_sessions =", CONFIG.get("debug_lvn_diag_sessions", 0))
for k, v in CONFIG.items():
    print(f"  {k:30s} = {v}")

print("\nCurrent default research config:")
for k in [
    "va_containment_threshold",
    "min_rejection_wick_ratio",
    "use_va_containment_filter",
    "use_poc_magnet_filter",
    "min_lvn_height_pct",
    "min_edge_bps",
]:
    print(f"  {k:30s} = {CONFIG[k]}")



Config loaded (v7_1H_structure_15m_execution)
  symbol                         = BTCUSDT
  path_1h                        = ../data/btc_1h.csv
  path_15m                       = ../data/btc_15m_perps.csv
  anchor_type                    = Daily
  fixed_bars                     = 100
  num_rows                       = 24
  value_area_pct                 = 70
  lvn_percentile                 = 30
  min_lvn_height_pct             = 0.5
  min_session_maturity_pct       = 30
  max_lvn_boxes                  = 10
  zone_max_age_sessions          = 3
  entry_mode                     = zone_edge
  inner_offset_bps               = 2
  order_ttl_bars                 = 12
  one_order_at_a_time            = True
  require_rejection              = False
  tp_target                      = poc
  sl_buffer_bps                  = 10
  max_hold_bars_15m              = 24
  min_edge_bps                   = 0
  conservative_same_bar_tp       = True
  fee_bps                        = 0
  slip_bps          

In [24]:
CONFIG["path_1h"] = "data/btc_1h.csv"
CONFIG["path_15m"] = "data/btc_15m_perps.csv"

assert os.path.exists(CONFIG["path_1h"]), f"Missing 1H file: {CONFIG['path_1h']}"
assert os.path.exists(CONFIG["path_15m"]), f"Missing 15m file: {CONFIG['path_15m']}"

print("1h exists:", os.path.exists(CONFIG["path_1h"]))
print("15m exists:", os.path.exists(CONFIG["path_15m"]))



1h exists: True
15m exists: True


## Cell 2 — Load Both Datasets (1H + 15m)

In [25]:
import os
print(os.getcwd())

C:\Users\olivi\lvn-auction-backtester


In [26]:
import os
import glob

print("CWD:", os.getcwd())
print("Data folder exists:", os.path.exists("data"))
print("Files in repo root:", os.listdir("."))
print("Files in data folder:", os.listdir("data") if os.path.exists("data") else "NO DATA FOLDER")
print("CSV matches root:", glob.glob("*.csv"))
print("CSV matches data:", glob.glob("data/*.csv"))

CWD: C:\Users\olivi\lvn-auction-backtester
Data folder exists: True
Files in repo root: ['.git', '.ipynb_checkpoints', 'data', 'lvn_limit_order_backtest (7).ipynb', 'README.md']
Files in data folder: ['btc_15m_perps.csv', 'btc_1h.csv']
CSV matches root: []
CSV matches data: ['data\\btc_15m_perps.csv', 'data\\btc_1h.csv']


In [27]:
print(CONFIG["path_1h"])
print(CONFIG["path_15m"])

data/btc_1h.csv
data/btc_15m_perps.csv


In [28]:
os.listdir("data")
print(CONFIG["path_1h"], CONFIG["path_15m"])

data/btc_1h.csv data/btc_15m_perps.csv


In [29]:
def load_ohlcv(path: str) -> pd.DataFrame:
    """Load OHLCV CSV and return a clean DataFrame indexed by UTC timestamp."""
    df = pd.read_csv(path)
    ts_col = None
    for c in df.columns:
        if c.lower() in ("timestamp", "datetime", "date", "time", "ts"):
            ts_col = c
            break
    if ts_col is None:
        ts_col = df.columns[0]

    df[ts_col] = pd.to_datetime(df[ts_col], utc=True)
    df = df.set_index(ts_col).sort_index()
    df.columns = [c.strip().lower() for c in df.columns]
    required = {"open", "high", "low", "close", "volume"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns: {missing}")
    df = df[["open", "high", "low", "close", "volume"]].copy()
    df = df.apply(pd.to_numeric, errors="coerce").dropna()
    print(f"Loaded {len(df)} bars  |  {df.index[0]} -> {df.index[-1]}")
    return df


# -- Load 1H (structure) --
assert os.path.exists(CONFIG["path_1h"]), f"Missing 1H file: {CONFIG['path_1h']}"
df_1h = load_ohlcv(CONFIG["path_1h"])

# -- Load 15m (execution) --
assert os.path.exists(CONFIG["path_15m"]), f"Missing 15m file: {CONFIG['path_15m']}"
df_15m = load_ohlcv(CONFIG["path_15m"])

# -- Align 15m bars to 1H timestamps --
df_15m["h1_key"] = df_15m.index.floor("1h")
h1_index_map = {ts: i for i, ts in enumerate(df_1h.index)}

# Report coverage
overlap_start = max(df_1h.index[0], df_15m.index[0])
overlap_end = min(df_1h.index[-1], df_15m.index[-1])
n_15m_aligned = df_15m["h1_key"].isin(h1_index_map).sum()
print(f"\n1H : {len(df_1h)} bars")
print(f"15m: {len(df_15m)} bars ({n_15m_aligned} aligned to 1H)")
print(f"Overlap: {overlap_start} -> {overlap_end}")


Loaded 54844 bars  |  2019-09-08 17:00:00+00:00 -> 2025-12-10 20:00:00+00:00
Loaded 226252 bars  |  2019-09-08 17:45:00+00:00 -> 2026-02-20 12:30:00+00:00

1H : 54844 bars
15m: 226252 bars (219373 aligned to 1H)
Overlap: 2019-09-08 17:45:00+00:00 -> 2025-12-10 20:00:00+00:00


## Cell 3 — Session Tagging (1H only)

In [30]:
def make_session_id(df: pd.DataFrame, anchor_type: str, fixed_bars: int = 100) -> pd.Series:
    """Assign integer session_id to each bar (applied to 1H data)."""
    idx = df.index
    if hasattr(idx, "tz") and idx.tz is not None:
        idx_utc = idx.tz_convert("UTC")
    else:
        idx_utc = idx

    if anchor_type == "Daily":
        raw = pd.Series(idx_utc.date, index=df.index)
    elif anchor_type == "Weekly":
        iso = idx_utc.isocalendar()
        raw = iso["year"].astype(str) + "-W" + iso["week"].astype(str).str.zfill(2)
        raw.index = df.index
    elif anchor_type == "Fixed Bars":
        raw = pd.Series(np.arange(len(df)) // fixed_bars, index=df.index)
    else:
        raise ValueError(f"Unknown anchor_type: {anchor_type!r}")

    labels = raw.unique()
    label_to_id = {lab: i for i, lab in enumerate(labels)}
    session_ids = raw.map(label_to_id).astype(int)
    return session_ids


df_1h["session_id"] = make_session_id(df_1h, CONFIG["anchor_type"], CONFIG["fixed_bars"])
print(f"1H sessions: {df_1h['session_id'].nunique()}")
print(df_1h.groupby("session_id").size().describe().to_string())

1H sessions: 2286
count    2286.000000
mean       23.991251
std         0.361025
min         7.000000
25%        24.000000
50%        24.000000
75%        24.000000
max        24.000000


## Cell 4 / 5 / 6 — Data Structures, POC/Value Area & Incremental Helpers

Same as v6: locked range, overlap volume, developing POC/VA, LVN detection, zone persistence metadata.
These operate on 1H data only.

In [31]:
# -- Data structures ------------------------------------------------------

@dataclass
class LVNZone:
    session_key: int
    start_bin: int
    end_bin: int
    bottom: float
    top: float
    session_key_created: int = 0
    poc_at_creation: float = 0.0


@dataclass
class SessionState:
    session_key: int
    session_low: float
    session_high: float
    row_height: float
    num_rows: int
    price_levels: np.ndarray
    bin_lows: np.ndarray
    bin_highs: np.ndarray
    volume_at_price: np.ndarray
    bar_count: int
    zones: List[LVNZone] = field(default_factory=list)
    poc_idx: int = 0
    poc_level: float = 0.0
    vah: float = 0.0
    val: float = 0.0
    poc_history: List[int] = field(default_factory=list)


# -- 1H state container for dual-TF use --

@dataclass
class StructureState1H:
    """Aggregates everything the 15m execution loop needs from the 1H engine."""
    session: Optional[SessionState] = None
    active_zones: List[LVNZone] = field(default_factory=list)
    poc: float = np.nan
    val: float = np.nan
    vah: float = np.nan
    prev_val: float = np.nan
    prev_vah: float = np.nan
    va_accept: float = np.nan
    current_sk: int = -1


def init_session(session_key: int, first_high: float, first_low: float,
                 num_rows: int) -> Optional[SessionState]:
    price_range = first_high - first_low
    if not np.isfinite(price_range) or price_range <= 0:
        return None
    row_height = price_range / num_rows
    price_levels = np.array(
        [first_low + row_height * i + row_height / 2.0 for i in range(num_rows)],
        dtype=np.float64,
    )
    return SessionState(
        session_key=session_key,
        session_low=first_low, session_high=first_high,
        row_height=row_height, num_rows=num_rows,
        price_levels=price_levels,
        bin_lows=price_levels - row_height / 2.0,
        bin_highs=price_levels + row_height / 2.0,
        volume_at_price=np.zeros(num_rows, dtype=np.float64),
        bar_count=1,
    )


def update_volume(state: SessionState, bar_low: float, bar_high: float,
                  bar_vol: float) -> None:
    state.bar_count += 1
    if not (np.isfinite(bar_low) and np.isfinite(bar_high) and np.isfinite(bar_vol)):
        return
    if bar_vol <= 0 or bar_high < state.session_low or bar_low > state.session_high:
        return
    denom = bar_high - bar_low
    if denom <= 0:
        return
    overlap_low = np.maximum(bar_low, state.bin_lows)
    overlap_high = np.minimum(bar_high, state.bin_highs)
    overlap = np.maximum(0.0, overlap_high - overlap_low)
    state.volume_at_price += bar_vol * (overlap / denom)


def compute_poc_and_value_area(
    volume_at_price: np.ndarray, price_levels: np.ndarray,
    num_rows: int, value_area_pct: float = 70.0,
) -> Tuple[int, float, float, float]:
    total = float(volume_at_price.sum())
    if total <= 0:
        return 0, np.nan, np.nan, np.nan
    poc_idx = int(np.argmax(volume_at_price))
    target = total * (value_area_pct / 100.0)
    va_lo = va_hi = poc_idx
    accum = float(volume_at_price[poc_idx])
    while accum < target:
        can_up = (va_hi + 1) < num_rows
        can_dn = (va_lo - 1) >= 0
        if not can_up and not can_dn:
            break
        v_up = float(volume_at_price[va_hi + 1]) if can_up else -np.inf
        v_dn = float(volume_at_price[va_lo - 1]) if can_dn else -np.inf
        if v_up >= v_dn:
            va_hi += 1; accum += v_up
        else:
            va_lo -= 1; accum += v_dn
    return poc_idx, float(price_levels[poc_idx]), float(price_levels[va_lo]), float(price_levels[va_hi])


def update_session_derived(state: SessionState, value_area_pct: float) -> None:
    poc_idx, poc_price, val_price, vah_price = compute_poc_and_value_area(
        state.volume_at_price, state.price_levels, state.num_rows, value_area_pct
    )
    state.poc_idx = poc_idx
    state.poc_level = poc_price
    state.val = val_price
    state.vah = vah_price
    state.poc_history.append(poc_idx)


def compute_min_lvn_rows(min_lvn_height_pct: float, num_rows: int) -> int:
    """Interpret min_lvn_height_pct as a literal percent of profile height."""
    return max(1, int(math.ceil((min_lvn_height_pct / 100.0) * num_rows)))


def detect_lvns(
    state: SessionState, lvn_percentile: float, min_lvn_height_pct: float,
    min_session_maturity_pct: float, max_lvn_boxes: int,
    debug_lvn_diag_sessions: int = 0, debug_lvn_diag_tracker: Optional[set] = None,
) -> Tuple[List[LVNZone], int]:
    session_length = state.bar_count
    min_bars = max(10, int(round(session_length * (min_session_maturity_pct / 100.0))))
    min_rows = compute_min_lvn_rows(min_lvn_height_pct, state.num_rows)
    if session_length < min_bars:
        return [], min_rows
    total = float(state.volume_at_price.sum())
    if total <= 0:
        return [], min_rows
    normalized = state.volume_at_price / total
    thresh = float(np.percentile(normalized, lvn_percentile))
    if (
        debug_lvn_diag_sessions > 0
        and debug_lvn_diag_tracker is not None
        and state.session_key not in debug_lvn_diag_tracker
        and len(debug_lvn_diag_tracker) < debug_lvn_diag_sessions
    ):
        print(
            f"[LVN diag] session={state.session_key} min_lvn_height_pct={min_lvn_height_pct:.4f} "
            f"computed_min_lvn_rows={min_rows} num_rows={state.num_rows}"
        )
        debug_lvn_diag_tracker.add(state.session_key)
    zones = []
    in_run = False; run_start = 0
    for i in range(state.num_rows):
        below = normalized[i] < thresh
        if below and not in_run:
            in_run = True; run_start = i
        is_last = (i == state.num_rows - 1)
        if in_run and (not below or is_last):
            run_end = i if (is_last and below) else (i - 1)
            if (run_end - run_start + 1) >= min_rows:
                zones.append(LVNZone(
                    session_key=state.session_key, start_bin=run_start, end_bin=run_end,
                    bottom=float(state.price_levels[run_start] - state.row_height / 2.0),
                    top=float(state.price_levels[run_end] + state.row_height / 2.0),
                    session_key_created=state.session_key,
                    poc_at_creation=state.poc_level,
                ))
            in_run = False
    if len(zones) > max_lvn_boxes:
        zones = zones[-max_lvn_boxes:]
    return zones, min_rows


def zone_key(z: LVNZone) -> Tuple:
    return (z.session_key_created, round(z.bottom, 8), round(z.top, 8))


print("Data structures & helpers loaded (v7).")

Data structures & helpers loaded (v7).


## Cell 7 — Dual-TF Backtest: 1H Structure + 15m Execution

**Architecture:**
1. Main loop iterates 15m bars.
2. When the 1H timestamp changes, advance the 1H engine (update volume, POC, VA, LVNs, va_accept).
3. On each 15m bar: check regime filter (va_accept), scan active zones for rejection, manage orders/exits.
4. Rejections are detected on 15m candles — much higher resolution than 1H.
5. Stops are placed relative to the 15m rejection bar's extreme.
6. TP targets the 1H developing POC.

In [32]:
# -- Trade data structures -------------------------------------------------

@dataclass
class PendingOrder:
    side: str
    price: float
    created_i: int       # index in 15m df
    ttl: int
    zone: LVNZone
    tp_price: float = 0.0
    sl_price: float = 0.0


@dataclass
class FilledTrade:
    entry_idx: int
    entry_time: object
    side: str
    entry_price: float
    zone_bottom: float
    zone_top: float
    zone_session_created: int
    tp_price: float
    sl_price: float
    poc_at_entry: float
    val_at_entry: float
    vah_at_entry: float
    va_accept_at_entry: float
    exit_idx: Optional[int] = None
    exit_time: Optional[object] = None
    exit_price: Optional[float] = None
    exit_reason: Optional[str] = None
    raw_pnl: Optional[float] = None
    net_pnl: Optional[float] = None
    pnl: Optional[float] = None


@dataclass
class RejectionSignal:
    side: str
    zone: LVNZone
    armed_bar: int       # 15m bar index
    rej_bar_low: float
    rej_bar_high: float


# -- Helpers --------------------------------------------------------------

def choose_entry_price(zone: LVNZone, mode: str, inner_offset_bps: float) -> Tuple[float, float]:
    mid = (zone.bottom + zone.top) / 2.0
    if mode == "zone_mid": return mid, mid
    if mode == "zone_edge": return zone.bottom, zone.top
    if mode == "zone_inner":
        return zone.bottom * (1.0 + inner_offset_bps / 10_000.0), zone.top * (1.0 - inner_offset_bps / 10_000.0)
    raise ValueError(f"Unknown entry_mode: {mode!r}")


def close_trade(trade: FilledTrade, idx: int, time, price: float, reason: str,
                fee_bps: float = 0, slip_bps: float = 0):
    adj = price * (1.0 - slip_bps / 10_000.0) if trade.side == "long" else price * (1.0 + slip_bps / 10_000.0)
    trade.exit_idx = idx
    trade.exit_time = time
    trade.exit_price = adj
    trade.exit_reason = reason

    raw_pnl = (adj - trade.entry_price) if trade.side == "long" else (trade.entry_price - adj)
    fee_cost = trade.entry_price * (fee_bps / 10_000.0) if fee_bps > 0 else 0.0
    net_pnl = raw_pnl - fee_cost

    trade.raw_pnl = raw_pnl
    trade.net_pnl = net_pnl
    trade.pnl = net_pnl




def compute_rejection_wick_ratio(side: str, zone: LVNZone, bar_low: float, bar_high: float) -> float:
    """Compute rejection strength as zone-penetration wick / full candle range."""
    bar_range = bar_high - bar_low
    if bar_range <= 0:
        return 0.0
    if side == "buy":
        penetration = max(0.0, zone.top - bar_low)
    elif side == "sell":
        penetration = max(0.0, bar_high - zone.bottom)
    else:
        return 0.0
    return penetration / bar_range

def evaluate_trade_economics(side: str, entry_price: float, target_price: float) -> Tuple[bool, Optional[float]]:
    if not (np.isfinite(entry_price) and np.isfinite(target_price)) or entry_price <= 0:
        return False, None
    if side == "long":
        favorable = target_price > entry_price
        edge_bps = ((target_price - entry_price) / entry_price) * 10_000.0
    else:
        favorable = target_price < entry_price
        edge_bps = ((entry_price - target_price) / entry_price) * 10_000.0
    return favorable, edge_bps


# -- 1H Structure Engine ---------------------------------------------------

def advance_1h_engine(
    st: StructureState1H,
    h1_bar,
    h1_session_id: int,
    cfg: dict,
    recent_1h_closes: deque,
    zone_keys_seen: set,
    lvn_audit: dict,
    debug_lvn_diag_sessions: int = 0,
    debug_lvn_diag_tracker: Optional[set] = None,
) -> None:
    """
    Advance the 1H structural state by one bar.
    Called once per hour when the 1H timestamp changes.
    """
    num_rows = cfg["num_rows"]
    va_pct = cfg["value_area_pct"]
    zone_max_age = cfg["zone_max_age_sessions"]

    sk = h1_session_id
    is_new_session = (sk != st.current_sk)

    if is_new_session:
        # Freeze outgoing session's VA
        if st.session is not None:
            if not np.isnan(st.session.val) and not np.isnan(st.session.vah):
                st.prev_val = st.session.val
                st.prev_vah = st.session.vah
            # Finalize zones into pool
            for z in st.session.zones:
                zk = zone_key(z)
                if zk not in zone_keys_seen:
                    zone_keys_seen.add(zk)
                    st.active_zones.append(z)
                    lvn_audit["total_active_zones_added"] += 1

        # Prune stale zones
        st.active_zones = [z for z in st.active_zones if (sk - z.session_key_created) <= zone_max_age]

        # Init new session
        st.current_sk = sk
        new_sess = init_session(sk, float(h1_bar["high"]), float(h1_bar["low"]), num_rows)
        st.session = new_sess
        if new_sess is not None:
            lvn_audit["sessions_processed"] += 1
            lvn_audit["session_zone_max"].setdefault(sk, 0)
            lvn_audit["session_sample"].setdefault(sk, {
                "session_id": sk,
                "num_rows": new_sess.num_rows,
                "min_lvn_height_pct": cfg["min_lvn_height_pct"],
                "computed_min_lvn_rows": compute_min_lvn_rows(cfg["min_lvn_height_pct"], new_sess.num_rows),
                "zones_created_this_session": 0,
            })
        recent_1h_closes.append(float(h1_bar["close"]))
        return  # first bar of session: no volume update

    if st.session is None:
        recent_1h_closes.append(float(h1_bar["close"]))
        return

    # Update developing profile
    update_volume(st.session, float(h1_bar["low"]), float(h1_bar["high"]), float(h1_bar["volume"]))
    update_session_derived(st.session, va_pct)
    st.session.zones, computed_min_rows = detect_lvns(
        st.session, cfg["lvn_percentile"], cfg["min_lvn_height_pct"],
        cfg["min_session_maturity_pct"], cfg["max_lvn_boxes"],
        debug_lvn_diag_sessions=debug_lvn_diag_sessions,
        debug_lvn_diag_tracker=debug_lvn_diag_tracker,
    )

    sk_sample = lvn_audit["session_sample"].setdefault(st.session.session_key, {
        "session_id": st.session.session_key,
        "num_rows": st.session.num_rows,
        "min_lvn_height_pct": cfg["min_lvn_height_pct"],
        "computed_min_lvn_rows": computed_min_rows,
        "zones_created_this_session": 0,
    })
    sk_sample["computed_min_lvn_rows"] = computed_min_rows
    lvn_audit["session_zone_max"][st.session.session_key] = max(
        lvn_audit["session_zone_max"].get(st.session.session_key, 0),
        len(st.session.zones),
    )
    sk_sample["zones_created_this_session"] = lvn_audit["session_zone_max"][st.session.session_key]

    # Merge new zones into pool
    for z in st.session.zones:
        zk = zone_key(z)
        if zk not in zone_keys_seen:
            zone_keys_seen.add(zk)
            st.active_zones.append(z)
            lvn_audit["total_active_zones_added"] += 1

    # Prune
    st.active_zones = [z for z in st.active_zones if (sk - z.session_key_created) <= zone_max_age]
    lvn_audit["max_simultaneous_active_zones"] = max(lvn_audit["max_simultaneous_active_zones"], len(st.active_zones))

    # Update derived
    st.poc = st.session.poc_level
    st.val = st.session.val
    st.vah = st.session.vah

    # VA acceptance against prior session's VA
    st.va_accept = np.nan
    if not np.isnan(st.prev_val) and not np.isnan(st.prev_vah):
        lookback = cfg["va_containment_lookback"]
        if len(recent_1h_closes) >= max(3, lookback // 2):
            inside = sum(1 for c in recent_1h_closes if st.prev_val <= c <= st.prev_vah)
            st.va_accept = inside / len(recent_1h_closes)

    recent_1h_closes.append(float(h1_bar["close"]))


# -- Main Dual-TF Backtest Loop -------------------------------------------

def backtest_dual_tf(
    df_1h: pd.DataFrame,
    df_15m: pd.DataFrame,
    h1_index_map: Dict,
    cfg: dict,
) -> Tuple[pd.DataFrame, StructureState1H, Dict[str, int], List[Dict], Dict[str, int], List[Dict], List[Dict], Dict[str, object]]:
    """
    1H structure + 15m execution backtest.
    Main loop iterates 15m bars.
    1H engine advances when the hour changes.
    """
    inner_bps = cfg["inner_offset_bps"]
    ttl = cfg["order_ttl_bars"]
    sl_buffer_bps = cfg["sl_buffer_bps"]
    max_hold = cfg["max_hold_bars_15m"]
    min_edge_bps = cfg["min_edge_bps"]
    conservative_tp = cfg.get("conservative_same_bar_tp", True)
    fee_bps = cfg.get("fee_bps", 0)
    slip_bps = cfg.get("slip_bps", 0)
    use_poc_filter = cfg.get("use_poc_magnet_filter", False)
    use_va_filter = cfg.get("use_va_containment_filter", False)
    va_threshold = cfg.get("va_containment_threshold", 0.5)
    require_rejection = cfg.get("require_rejection", True)
    min_rej_wick_ratio = cfg.get("min_rejection_wick_ratio", 0.0)
    use_rejection_wick_ratio_filter = min_rej_wick_ratio > 0
    debug_lvn_diag_sessions = int(cfg.get("debug_lvn_diag_sessions", 0) or 0)
    debug_lvn_diag_tracker = set() if debug_lvn_diag_sessions > 0 else None

    if cfg.get("debug_disable_va_filter", False):
        use_va_filter = False
    if cfg.get("debug_disable_rejection", False):
        require_rejection = False
    if cfg.get("debug_disable_magnet_filter", False):
        use_poc_filter = False
    if cfg.get("debug_min_edge_bps_override") is not None:
        min_edge_bps = float(cfg.get("debug_min_edge_bps_override"))

    debug_counts = {
        "same_bar_entry_tp_touched": 0,
        "econ_filter_checked": 0,
        "econ_filter_pass": 0,
        "econ_filter_rejected": 0,
        "bars_15m_seen": 0,
        "h1_updates": 0,
        "have_session": 0,
        "have_active_zones": 0,
        "zone_overlap": 0,
        "va_filter_checked": 0,
        "va_pass": 0,
        "magnet_filter_checked": 0,
        "magnet_pass": 0,
        "edge_filter_checked": 0,
        "edge_pass": 0,
        "rejection_filter_checked": 0,
        "rejection_pass": 0,
        "rejection_wick_fail": 0,
        "rejection_wick_bypass": 0,
        "pending_orders_created": 0,
        "orders_filled": 0,
        "orders_expired": 0,
        "orders_cancelled": 0,
    }
    skip_samples: List[Dict] = []
    max_skip_samples = 30
    candidate_samples: List[Dict] = []
    max_candidate_samples = 20
    same_bar_examples: List[Dict] = []

    def add_candidate_sample(ts_15m, side, zone, entry_price, poc_price, edge_bps, favorable_target_bool):
        if len(candidate_samples) >= max_candidate_samples:
            return
        candidate_samples.append({
            "ts_15m": ts_15m,
            "side": side,
            "zone_bottom": float(zone.bottom),
            "zone_top": float(zone.top),
            "entry_price": float(entry_price),
            "poc_price": float(poc_price) if np.isfinite(poc_price) else np.nan,
            "edge_bps": float(edge_bps) if edge_bps is not None else np.nan,
            "favorable_target_bool": bool(favorable_target_bool),
        })

    def add_skip_sample(ts_15m, zone, close, va_accept, poc_price, edge_bps, failed_gate):
        if len(skip_samples) >= max_skip_samples:
            return
        skip_samples.append({
            "ts_15m": ts_15m,
            "zone_bottom": float(zone.bottom),
            "zone_top": float(zone.top),
            "close": float(close),
            "va_accept": float(va_accept) if not np.isnan(va_accept) else np.nan,
            "poc_price": float(poc_price) if not np.isnan(poc_price) else np.nan,
            "edge_bps": float(edge_bps) if edge_bps is not None else np.nan,
            "failed_gate": failed_gate,
        })

    # 1H state
    st = StructureState1H()
    recent_1h_closes: deque = deque(maxlen=cfg["va_containment_lookback"])
    zone_keys_seen: set = set()

    lvn_audit = {
        "sessions_processed": 0,
        "session_zone_max": {},
        "session_sample": {},
        "total_active_zones_added": 0,
        "max_simultaneous_active_zones": 0,
    }

    # 15m execution state
    pos = 0
    order: Optional[PendingOrder] = None
    trades: List[FilledTrade] = []
    active_trade: Optional[FilledTrade] = None
    rejection: Optional[RejectionSignal] = None

    # Pre-extract 15m arrays
    m15_highs = df_15m["high"].values
    m15_lows = df_15m["low"].values
    m15_closes = df_15m["close"].values
    m15_opens = df_15m["open"].values
    m15_times = df_15m.index
    m15_h1keys = df_15m["h1_key"].to_numpy(dtype=object)
    h1_session_ids = df_1h["session_id"].values

    current_h1_ts = None
    n = len(df_15m)
    h1_tz = df_1h.index.tz
    debug_15m_limit = int(cfg.get("debug_h1_trace_bars", 0))

    def normalize_h1_key(raw_ts):
        ts = pd.Timestamp(raw_ts)
        if h1_tz is not None:
            if ts.tzinfo is None:
                ts = ts.tz_localize(h1_tz)
            else:
                ts = ts.tz_convert(h1_tz)
        elif ts.tzinfo is not None:
            ts = ts.tz_convert("UTC").tz_localize(None)
        return ts

    for i in range(n):
        debug_counts["bars_15m_seen"] += 1
        raw_h1_ts = m15_h1keys[i]
        h1_ts = normalize_h1_key(raw_h1_ts)
        pre_current_h1_ts = current_h1_ts
        entered_h1_update = False

        # Skip 15m bars that don't map to any 1H bar
        if h1_ts not in h1_index_map:
            if i < debug_15m_limit:
                print(f"[H1 TRACE] i={i} ts_15m={m15_times[i]} h1_key={h1_ts} current_before={pre_current_h1_ts} entered=False mapped=False session_exists={st.session is not None}")
            continue

        # ================================================================
        # ADVANCE 1H ENGINE on hour change
        # ================================================================
        if current_h1_ts is None or h1_ts != current_h1_ts:
            entered_h1_update = True
            h1_i = h1_index_map[h1_ts]
            h1_bar = df_1h.iloc[h1_i]
            h1_sk = int(h1_session_ids[h1_i])

            # If session changed, force time_stop (not session_end)
            if h1_sk != st.current_sk and st.current_sk != -1:
                if pos != 0 and active_trade is not None:
                    close_trade(active_trade, i, m15_times[i], float(m15_opens[i]),
                                "time_stop", fee_bps, slip_bps)
                    pos = 0; active_trade = None
                    if order is not None:
                        debug_counts["orders_cancelled"] += 1
                        order = None
                rejection = None

            advance_1h_engine(
                st, h1_bar, h1_sk, cfg, recent_1h_closes, zone_keys_seen, lvn_audit,
                debug_lvn_diag_sessions=debug_lvn_diag_sessions,
                debug_lvn_diag_tracker=debug_lvn_diag_tracker,
            )
            debug_counts["h1_updates"] += 1
            current_h1_ts = h1_ts

        if i < debug_15m_limit:
            print(f"[H1 TRACE] i={i} ts_15m={m15_times[i]} h1_key={h1_ts} current_before={pre_current_h1_ts} entered={entered_h1_update} mapped=True session_exists={st.session is not None}")

        if st.session is None:
            continue

        debug_counts["have_session"] += 1
        poc = st.poc
        tradeable_zones = st.active_zones
        if tradeable_zones:
            debug_counts["have_active_zones"] += 1

        # ================================================================
        # EXIT LOGIC (15m)
        # ================================================================
        if pos != 0 and active_trade is not None:
            exited = False
            is_entry_bar = (i == active_trade.entry_idx)

            # TP at 1H POC
            if not exited and not np.isnan(poc) and not (conservative_tp and is_entry_bar):
                if pos == 1 and m15_highs[i] >= active_trade.tp_price:
                    close_trade(active_trade, i, m15_times[i],
                                active_trade.tp_price, "tp_poc", fee_bps, slip_bps)
                    exited = True
                elif pos == -1 and m15_lows[i] <= active_trade.tp_price:
                    close_trade(active_trade, i, m15_times[i],
                                active_trade.tp_price, "tp_poc", fee_bps, slip_bps)
                    exited = True

            # SL (rejection invalidation)
            if not exited:
                if pos == 1 and m15_lows[i] <= active_trade.sl_price:
                    close_trade(active_trade, i, m15_times[i],
                                active_trade.sl_price, "sl_rej", fee_bps, slip_bps)
                    exited = True
                elif pos == -1 and m15_highs[i] >= active_trade.sl_price:
                    close_trade(active_trade, i, m15_times[i],
                                active_trade.sl_price, "sl_rej", fee_bps, slip_bps)
                    exited = True

            # Time stop (15m bars)
            if not exited and (i - active_trade.entry_idx) >= max_hold:
                close_trade(active_trade, i, m15_times[i],
                            float(m15_closes[i]), "time_stop", fee_bps, slip_bps)
                exited = True

            if exited:
                pos = 0; active_trade = None

        # ================================================================
        # ORDER MANAGEMENT (15m)
        # ================================================================
        if order is not None and (i - order.created_i) >= order.ttl:
            debug_counts["orders_expired"] += 1
            order = None

        if order is not None and pos == 0:
            filled = False
            if order.side == "buy" and m15_lows[i] <= order.price:
                filled = True
                fill_px = order.price * (1.0 + slip_bps / 10_000.0)
                fill_poc = poc if not np.isnan(poc) else order.tp_price
                active_trade = FilledTrade(
                    entry_idx=i, entry_time=m15_times[i], side="long",
                    entry_price=float(fill_px),
                    zone_bottom=order.zone.bottom, zone_top=order.zone.top,
                    zone_session_created=order.zone.session_key_created,
                    tp_price=float(fill_poc), sl_price=float(order.sl_price),
                    poc_at_entry=float(poc) if not np.isnan(poc) else 0.0,
                    val_at_entry=float(st.val) if not np.isnan(st.val) else 0.0,
                    vah_at_entry=float(st.vah) if not np.isnan(st.vah) else 0.0,
                    va_accept_at_entry=float(st.va_accept) if not np.isnan(st.va_accept) else 0.0,
                )
                trades.append(active_trade); pos = 1
            elif order.side == "sell" and m15_highs[i] >= order.price:
                filled = True
                fill_px = order.price * (1.0 - slip_bps / 10_000.0)
                fill_poc = poc if not np.isnan(poc) else order.tp_price
                active_trade = FilledTrade(
                    entry_idx=i, entry_time=m15_times[i], side="short",
                    entry_price=float(fill_px),
                    zone_bottom=order.zone.bottom, zone_top=order.zone.top,
                    zone_session_created=order.zone.session_key_created,
                    tp_price=float(fill_poc), sl_price=float(order.sl_price),
                    poc_at_entry=float(poc) if not np.isnan(poc) else 0.0,
                    val_at_entry=float(st.val) if not np.isnan(st.val) else 0.0,
                    vah_at_entry=float(st.vah) if not np.isnan(st.vah) else 0.0,
                    va_accept_at_entry=float(st.va_accept) if not np.isnan(st.va_accept) else 0.0,
                )
                trades.append(active_trade); pos = -1
            if filled and active_trade is not None:
                same_bar_tp_hit = (
                    (active_trade.side == "long" and m15_highs[i] >= active_trade.tp_price)
                    or (active_trade.side == "short" and m15_lows[i] <= active_trade.tp_price)
                )
                if same_bar_tp_hit:
                    debug_counts["same_bar_entry_tp_touched"] += 1
                    if len(same_bar_examples) < 5:
                        same_bar_examples.append({
                            "entry_time": m15_times[i],
                            "side": active_trade.side,
                            "entry_price": float(active_trade.entry_price),
                            "tp_price": float(active_trade.tp_price),
                            "bar_high": float(m15_highs[i]),
                            "bar_low": float(m15_lows[i]),
                        })
            if filled:
                debug_counts["orders_filled"] += 1
                order = None

        # ================================================================
        # ENTRY LOGIC (15m rejection against 1H zones)
        # ================================================================
        if pos != 0 or i < 1:
            continue

        # -- Convert armed rejection into limit order --
        if rejection is not None and rejection.armed_bar == i - 1:
            if use_va_filter:
                debug_counts["va_filter_checked"] += 1
            if use_va_filter and (np.isnan(st.va_accept) or st.va_accept < va_threshold):
                add_skip_sample(m15_times[i], rejection.zone, m15_closes[i], st.va_accept, poc, None, "va_filter")
                rejection = None
            elif order is None or not cfg["one_order_at_a_time"]:
                if use_va_filter:
                    debug_counts["va_pass"] += 1
                rz = rejection.zone
                buy_px, sell_px = choose_entry_price(rz, cfg["entry_mode"], inner_bps)

                if rejection.side == "buy":
                    debug_counts["magnet_filter_checked"] += 1
                    if use_poc_filter and not np.isnan(poc) and poc <= buy_px:
                        add_skip_sample(m15_times[i], rz, m15_closes[i], st.va_accept, poc, abs(poc - buy_px) / buy_px * 10_000 if buy_px > 0 and not np.isnan(poc) else None, "magnet_filter")
                        rejection = None
                    else:
                        debug_counts["magnet_pass"] += 1
                        tp = poc if not np.isnan(poc) else rz.top + (rz.top - rz.bottom)
                        favorable_target, edge_bps = evaluate_trade_economics("long", buy_px, tp)
                        add_candidate_sample(m15_times[i], "long", rz, buy_px, tp, edge_bps, favorable_target)
                        debug_counts["econ_filter_checked"] += 1
                        if not favorable_target:
                            debug_counts["econ_filter_rejected"] += 1
                            add_skip_sample(m15_times[i], rz, m15_closes[i], st.va_accept, poc, edge_bps, "tp_not_favorable")
                            rejection = None
                            continue
                        debug_counts["econ_filter_pass"] += 1
                        debug_counts["edge_filter_checked"] += 1
                        if min_edge_bps > 0 and edge_bps is not None and edge_bps < min_edge_bps:
                            debug_counts["econ_filter_rejected"] += 1
                            add_skip_sample(m15_times[i], rz, m15_closes[i], st.va_accept, poc, edge_bps, "edge_filter")
                            rejection = None
                            continue
                        debug_counts["edge_pass"] += 1
                        sl = rejection.rej_bar_low * (1.0 - sl_buffer_bps / 10_000.0)
                        order = PendingOrder("buy", float(buy_px), i, ttl, rz, tp_price=float(tp), sl_price=float(sl))
                        debug_counts["pending_orders_created"] += 1
                        rejection = None
                elif rejection.side == "sell":
                    debug_counts["magnet_filter_checked"] += 1
                    if use_poc_filter and not np.isnan(poc) and poc >= sell_px:
                        add_skip_sample(m15_times[i], rz, m15_closes[i], st.va_accept, poc, abs(poc - sell_px) / sell_px * 10_000 if sell_px > 0 and not np.isnan(poc) else None, "magnet_filter")
                        rejection = None
                    else:
                        debug_counts["magnet_pass"] += 1
                        tp = poc if not np.isnan(poc) else rz.bottom - (rz.top - rz.bottom)
                        favorable_target, edge_bps = evaluate_trade_economics("short", sell_px, tp)
                        add_candidate_sample(m15_times[i], "short", rz, sell_px, tp, edge_bps, favorable_target)
                        debug_counts["econ_filter_checked"] += 1
                        if not favorable_target:
                            debug_counts["econ_filter_rejected"] += 1
                            add_skip_sample(m15_times[i], rz, m15_closes[i], st.va_accept, poc, edge_bps, "tp_not_favorable")
                            rejection = None
                            continue
                        debug_counts["econ_filter_pass"] += 1
                        debug_counts["edge_filter_checked"] += 1
                        if min_edge_bps > 0 and edge_bps is not None and edge_bps < min_edge_bps:
                            debug_counts["econ_filter_rejected"] += 1
                            add_skip_sample(m15_times[i], rz, m15_closes[i], st.va_accept, poc, edge_bps, "edge_filter")
                            rejection = None
                            continue
                        debug_counts["edge_pass"] += 1
                        sl = rejection.rej_bar_high * (1.0 + sl_buffer_bps / 10_000.0)
                        order = PendingOrder("sell", float(sell_px), i, ttl, rz, tp_price=float(tp), sl_price=float(sl))
                        debug_counts["pending_orders_created"] += 1
                        rejection = None
            else:
                rejection = None
        elif rejection is not None:
            rejection = None  # stale

        # -- Scan for new rejection on 15m bar --
        if order is None and rejection is None:
            va_pass = True
            if use_va_filter:
                debug_counts["va_filter_checked"] += 1
                if np.isnan(st.va_accept) or st.va_accept < va_threshold:
                    va_pass = False
                else:
                    debug_counts["va_pass"] += 1

            if va_pass:
                prev_close = float(m15_closes[i - 1])
                bar_low = float(m15_lows[i])
                bar_high = float(m15_highs[i])
                bar_close = float(m15_closes[i])

                for zone in tradeable_zones:
                    overlaps = (bar_low <= zone.top) and (bar_high >= zone.bottom)
                    if not overlaps:
                        continue
                    debug_counts["zone_overlap"] += 1

                    if prev_close > zone.top:
                        if require_rejection:
                            debug_counts["rejection_filter_checked"] += 1
                            if bar_close > zone.top:
                                if use_rejection_wick_ratio_filter:
                                    wick_ratio = compute_rejection_wick_ratio("buy", zone, bar_low, bar_high)
                                    if wick_ratio < min_rej_wick_ratio:
                                        debug_counts["rejection_wick_fail"] += 1
                                        add_skip_sample(m15_times[i], zone, bar_close, st.va_accept, poc, None, "rejection_wick_ratio")
                                        continue
                                else:
                                    debug_counts["rejection_wick_bypass"] += 1
                                debug_counts["rejection_pass"] += 1
                                rejection = RejectionSignal("buy", zone, i, rej_bar_low=bar_low, rej_bar_high=bar_high)
                                break
                            add_skip_sample(m15_times[i], zone, bar_close, st.va_accept, poc, None, "rejection_filter")
                        else:
                            buy_px, _ = choose_entry_price(zone, cfg["entry_mode"], inner_bps)
                            debug_counts["magnet_filter_checked"] += 1
                            if use_poc_filter and not np.isnan(poc) and poc <= buy_px:
                                add_skip_sample(m15_times[i], zone, bar_close, st.va_accept, poc, abs(poc - buy_px) / buy_px * 10_000 if buy_px > 0 and not np.isnan(poc) else None, "magnet_filter")
                                continue
                            debug_counts["magnet_pass"] += 1
                            tp = poc if not np.isnan(poc) else zone.top + (zone.top - zone.bottom)
                            favorable_target, edge_bps = evaluate_trade_economics("long", buy_px, tp)
                            add_candidate_sample(m15_times[i], "long", zone, buy_px, tp, edge_bps, favorable_target)
                            debug_counts["econ_filter_checked"] += 1
                            if not favorable_target:
                                debug_counts["econ_filter_rejected"] += 1
                                add_skip_sample(m15_times[i], zone, bar_close, st.va_accept, poc, edge_bps, "tp_not_favorable")
                                continue
                            debug_counts["econ_filter_pass"] += 1
                            debug_counts["edge_filter_checked"] += 1
                            if min_edge_bps > 0 and edge_bps is not None and edge_bps < min_edge_bps:
                                debug_counts["econ_filter_rejected"] += 1
                                add_skip_sample(m15_times[i], zone, bar_close, st.va_accept, poc, edge_bps, "edge_filter")
                                continue
                            debug_counts["edge_pass"] += 1
                            sl = bar_low * (1.0 - sl_buffer_bps / 10_000.0)
                            order = PendingOrder("buy", float(buy_px), i, ttl, zone, tp_price=float(tp), sl_price=float(sl))
                            debug_counts["pending_orders_created"] += 1
                            break

                    if prev_close < zone.bottom:
                        if require_rejection:
                            debug_counts["rejection_filter_checked"] += 1
                            if bar_close < zone.bottom:
                                if use_rejection_wick_ratio_filter:
                                    wick_ratio = compute_rejection_wick_ratio("sell", zone, bar_low, bar_high)
                                    if wick_ratio < min_rej_wick_ratio:
                                        debug_counts["rejection_wick_fail"] += 1
                                        add_skip_sample(m15_times[i], zone, bar_close, st.va_accept, poc, None, "rejection_wick_ratio")
                                        continue
                                else:
                                    debug_counts["rejection_wick_bypass"] += 1
                                debug_counts["rejection_pass"] += 1
                                rejection = RejectionSignal("sell", zone, i, rej_bar_low=bar_low, rej_bar_high=bar_high)
                                break
                            add_skip_sample(m15_times[i], zone, bar_close, st.va_accept, poc, None, "rejection_filter")
                        else:
                            _, sell_px = choose_entry_price(zone, cfg["entry_mode"], inner_bps)
                            debug_counts["magnet_filter_checked"] += 1
                            if use_poc_filter and not np.isnan(poc) and poc >= sell_px:
                                add_skip_sample(m15_times[i], zone, bar_close, st.va_accept, poc, abs(poc - sell_px) / sell_px * 10_000 if sell_px > 0 and not np.isnan(poc) else None, "magnet_filter")
                                continue
                            debug_counts["magnet_pass"] += 1
                            tp = poc if not np.isnan(poc) else zone.bottom - (zone.top - zone.bottom)
                            favorable_target, edge_bps = evaluate_trade_economics("short", sell_px, tp)
                            add_candidate_sample(m15_times[i], "short", zone, sell_px, tp, edge_bps, favorable_target)
                            debug_counts["econ_filter_checked"] += 1
                            if not favorable_target:
                                debug_counts["econ_filter_rejected"] += 1
                                add_skip_sample(m15_times[i], zone, bar_close, st.va_accept, poc, edge_bps, "tp_not_favorable")
                                continue
                            debug_counts["econ_filter_pass"] += 1
                            debug_counts["edge_filter_checked"] += 1
                            if min_edge_bps > 0 and edge_bps is not None and edge_bps < min_edge_bps:
                                debug_counts["econ_filter_rejected"] += 1
                                add_skip_sample(m15_times[i], zone, bar_close, st.va_accept, poc, edge_bps, "edge_filter")
                                continue
                            debug_counts["edge_pass"] += 1
                            sl = bar_high * (1.0 + sl_buffer_bps / 10_000.0)
                            order = PendingOrder("sell", float(sell_px), i, ttl, zone, tp_price=float(tp), sl_price=float(sl))
                            debug_counts["pending_orders_created"] += 1
                            break
    # -- Build results --
    session_zone_max = lvn_audit["session_zone_max"]
    valid_sessions = [sid for sid, zone_count in session_zone_max.items() if zone_count > 0]
    zones_per_valid_session = [session_zone_max[sid] for sid in valid_sessions]
    sample_rows = []
    for sid in sorted(lvn_audit["session_sample"].keys())[:5]:
        sample_rows.append(dict(lvn_audit["session_sample"][sid]))

    lvn_diagnostics = {
        "sessions_processed": int(lvn_audit["sessions_processed"]),
        "sessions_with_valid_lvns": int(len(valid_sessions)),
        "total_lvn_zones_created": int(sum(session_zone_max.values())),
        "total_active_zones_ever_added": int(lvn_audit["total_active_zones_added"]),
        "max_simultaneous_active_zones": int(lvn_audit["max_simultaneous_active_zones"]),
        "average_zones_per_valid_session": float(np.mean(zones_per_valid_session)) if zones_per_valid_session else 0.0,
        "sample_sessions": sample_rows,
    }

    zone_diagnostics = {
        "active_zones_end": len(st.active_zones),
        "zone_keys_seen": len(zone_keys_seen),
        "current_session_zone_count": len(st.session.zones) if st.session is not None else 0,
    }
    if not trades:
        print("No trades generated.")
        return pd.DataFrame(), st, debug_counts, skip_samples, zone_diagnostics, candidate_samples, same_bar_examples, lvn_diagnostics

    records = [{
        "entry_idx": t.entry_idx, "entry_time": t.entry_time,
        "side": t.side, "entry_price": t.entry_price,
        "zone_bottom": t.zone_bottom, "zone_top": t.zone_top,
        "zone_session_created": t.zone_session_created,
        "tp_price": t.tp_price, "sl_price": t.sl_price,
        "poc_at_entry": t.poc_at_entry,
        "val_at_entry": t.val_at_entry, "vah_at_entry": t.vah_at_entry,
        "va_accept_at_entry": t.va_accept_at_entry,
        "exit_idx": t.exit_idx, "exit_time": t.exit_time,
        "exit_price": t.exit_price, "exit_reason": t.exit_reason,
        "raw_pnl": t.raw_pnl, "net_pnl": t.net_pnl, "pnl": t.pnl,
    } for t in trades]
    return pd.DataFrame(records), st, debug_counts, skip_samples, zone_diagnostics, candidate_samples, same_bar_examples, lvn_diagnostics


# -- Run --
trades_df, final_state, debug_counts, skip_samples, zone_diagnostics, candidate_samples, same_bar_examples, lvn_diagnostics = backtest_dual_tf(df_1h, df_15m, h1_index_map, CONFIG)

print("\n=== Path diagnostics ===")
print("path_1h:", CONFIG["path_1h"], "exists=", os.path.exists(CONFIG["path_1h"]))
print("path_15m:", CONFIG["path_15m"], "exists=", os.path.exists(CONFIG["path_15m"]))

print("\n=== Runtime config values ===")
print("va_containment_threshold =", CONFIG["va_containment_threshold"])
print("min_edge_bps =", CONFIG["min_edge_bps"])
print("min_rejection_wick_ratio =", CONFIG["min_rejection_wick_ratio"])
print("min_lvn_height_pct =", CONFIG["min_lvn_height_pct"])
print("debug_h1_trace_bars =", CONFIG.get("debug_h1_trace_bars", 0))
print("debug_lvn_diag_sessions =", CONFIG.get("debug_lvn_diag_sessions", 0))
print("num_rows =", CONFIG["num_rows"])
print("computed_min_lvn_rows =", compute_min_lvn_rows(CONFIG["min_lvn_height_pct"], CONFIG["num_rows"]))

print("\n=== Debug counts ===")
for k, v in debug_counts.items():
    print(f"{k:28s}: {v}")

print("=== Stage diagnostics (critical path) ===")
critical_stage_keys = [
    "bars_15m_seen",
    "h1_updates",
    "have_session",
    "have_active_zones",
    "zone_overlap",
    "va_pass",
    "magnet_pass",
    "edge_pass",
    "rejection_pass",
    "pending_orders_created",
    "orders_filled",
]
for k in critical_stage_keys:
    print(f"{k:28s}: {debug_counts.get(k, 0)}")
print()

print("=== LVN creation diagnostics ===")
for k in [
    "sessions_processed",
    "sessions_with_valid_lvns",
    "total_lvn_zones_created",
    "total_active_zones_ever_added",
    "max_simultaneous_active_zones",
    "average_zones_per_valid_session",
]:
    print(f"{k:32s}: {lvn_diagnostics[k]}")
print("\nLVN sample sessions (first few):")
if lvn_diagnostics.get("sample_sessions"):
    display(pd.DataFrame(lvn_diagnostics["sample_sessions"]).head(5))
else:
    print("No LVN session samples captured.")

collapse_stage = "Unknown"
if lvn_diagnostics["sessions_processed"] > 0 and lvn_diagnostics["total_lvn_zones_created"] == 0:
    collapse_stage = "A. No LVNs are being created"
elif lvn_diagnostics["total_lvn_zones_created"] > 0 and debug_counts["have_active_zones"] == 0:
    collapse_stage = "B. LVNs exist, but no active zones persist"
elif debug_counts["have_active_zones"] > 0 and debug_counts["zone_overlap"] == 0:
    collapse_stage = "C. Active zones exist, but no 15m overlaps"
elif debug_counts["zone_overlap"] > 0 and debug_counts["pending_orders_created"] == 0:
    collapse_stage = "D. Overlaps happen, but entry filters reject everything"
elif debug_counts["pending_orders_created"] > 0 and debug_counts["orders_filled"] == 0:
    collapse_stage = "E. Orders are created, but never filled"
else:
    collapse_stage = "Pipeline not collapsed to zero at a single gate (orders/trades exist)"
print("\nPipeline collapse classification:", collapse_stage)
print()
print("\n=== Skip samples (first 30) ===")
skip_samples_df = pd.DataFrame(skip_samples)
if len(skip_samples_df):
    display(skip_samples_df)
else:
    print("No skipped candidates captured.")

print("\n=== Candidate economics samples (20) ===")
candidate_samples_df = pd.DataFrame(candidate_samples)
if len(candidate_samples_df):
    display(candidate_samples_df.head(20))
else:
    print("No candidate samples captured.")

print("\n=== Same-bar entry+tp touch examples ===")
if len(same_bar_examples):
    display(pd.DataFrame(same_bar_examples))
else:
    print("No same-bar entry+tp touch cases recorded.")

print("\n=== Zone lifecycle diagnostics ===")
print(zone_diagnostics)

print(f"\nFinal trades count: {len(trades_df)}")
if len(trades_df):
    print("Trades per year:")
    print(trades_df["entry_time"].dt.year.value_counts().sort_index().to_string())
    display(trades_df.head(10))

    closed = trades_df.dropna(subset=["exit_price"]).copy()
    closed["raw_pnl_check"] = np.where(closed["side"] == "long", closed["exit_price"] - closed["entry_price"], closed["entry_price"] - closed["exit_price"])
    closed["raw_pnl_ok"] = np.isclose(closed["raw_pnl"], closed["raw_pnl_check"], atol=1e-8)
    print("\n=== PnL math audit sample (10 trades) ===")
    pnl_cols = ["side", "entry_price", "exit_price", "exit_reason", "raw_pnl", "net_pnl", "raw_pnl_check", "raw_pnl_ok"]
    display(closed[pnl_cols].head(10))

    closed["tp_favorable"] = np.where(closed["side"] == "long", closed["tp_price"] > closed["entry_price"], closed["tp_price"] < closed["entry_price"])
    tp_poc = closed[closed["exit_reason"] == "tp_poc"].copy()
    favorable_tp_rate = float(tp_poc["tp_favorable"].mean()) if len(tp_poc) else np.nan
    print(f"favorable_tp_rate (tp_poc exits): {favorable_tp_rate:.4f}" if len(tp_poc) else "favorable_tp_rate (tp_poc exits): n/a")

    print("\n=== Avg pnl by exit_reason and side ===")
    summary_tbl = closed.groupby(["exit_reason", "side"], dropna=False)[["raw_pnl", "net_pnl"]].mean().round(4)
    display(summary_tbl)

    econ_cols = ["entry_time", "side", "entry_price", "tp_price", "sl_price", "exit_price", "exit_reason", "raw_pnl", "net_pnl", "zone_bottom", "zone_top"]
    print("\n=== Sample tp_poc trades (20) ===")
    display(closed.loc[closed["exit_reason"] == "tp_poc", econ_cols].head(20))

    print("\n=== Sample sl_rej trades (20) ===")
    display(closed.loc[closed["exit_reason"] == "sl_rej", econ_cols].head(20))




print("\n=== Structural sanity fallback (filters disabled) ===")
debug_config = CONFIG.copy()
debug_config["use_va_containment_filter"] = False
debug_config["use_poc_magnet_filter"] = False
debug_config["min_edge_bps"] = 0
debug_config["require_rejection"] = False

debug_trades_df, _, debug_counts_fallback, _, debug_zone_diag, _, _, debug_lvn_diag = backtest_dual_tf(
    df_1h, df_15m, h1_index_map, debug_config
)
print("fallback_trades =", len(debug_trades_df))
print("fallback_total_lvn_zones_created =", debug_lvn_diag["total_lvn_zones_created"])
print("fallback_total_active_zones_ever_added =", debug_lvn_diag["total_active_zones_ever_added"])
print("fallback_active_zones_end =", debug_zone_diag["active_zones_end"])
print("fallback_zone_overlap =", debug_counts_fallback["zone_overlap"])
print("fallback_orders_created =", debug_counts_fallback["pending_orders_created"])
print("fallback_orders_filled =", debug_counts_fallback["orders_filled"])
print("fallback_zone_diagnostics =", debug_zone_diag)


No trades generated.
Trades: 0


## Cell 8 — Results Summary & Diagnostics

**v7 targets:**
- Trade count: 200–400 (up from ~60–84).
- `sl_rej` < 50%.
- Avg hold: 2–6 hours (in 15m bars: 8–24).
- Stable PF across split halves.

In [33]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

def compute_profit_factor(closed: pd.DataFrame) -> float:
    if closed is None or len(closed) == 0:
        return np.nan
    gross_profit = closed.loc[closed["pnl"] > 0, "pnl"].sum()
    gross_loss = -closed.loc[closed["pnl"] < 0, "pnl"].sum()
    return float(gross_profit / gross_loss) if gross_loss > 0 else np.nan


def compute_trade_metrics(tdf: pd.DataFrame) -> Dict[str, float]:
    if tdf is None or len(tdf) == 0:
        return {
            "Trades": 0,
            "Win rate": np.nan,
            "Profit factor": np.nan,
            "Avg PnL per trade": np.nan,
            "Total PnL": np.nan,
            "sl_rej percentage": np.nan,
            "tp_poc percentage": np.nan,
            "Avg hold (hours)": np.nan,
        }

    closed = tdf.dropna(subset=["pnl"]).copy()
    if len(closed) == 0:
        return {
            "Trades": int(len(tdf)),
            "Win rate": np.nan,
            "Profit factor": np.nan,
            "Avg PnL per trade": np.nan,
            "Total PnL": np.nan,
            "sl_rej percentage": np.nan,
            "tp_poc percentage": np.nan,
            "Avg hold (hours)": np.nan,
        }

    wins = int((closed["pnl"] > 0).sum())
    closed["hold_bars"] = closed["exit_idx"] - closed["entry_idx"]
    exit_counts = closed["exit_reason"].value_counts(normalize=True) * 100.0

    return {
        "Trades": int(len(closed)),
        "Win rate": float((wins / len(closed)) * 100.0),
        "Profit factor": compute_profit_factor(closed),
        "Avg PnL per trade": float(closed["pnl"].mean()),
        "Total PnL": float(closed["pnl"].sum()),
        "sl_rej percentage": float(exit_counts.get("sl_rej", 0.0)),
        "tp_poc percentage": float(exit_counts.get("tp_poc", 0.0)),
        "Avg hold (hours)": float(closed["hold_bars"].mean() * 0.25),
    }


def compute_split_profit_factors(tdf: pd.DataFrame) -> Tuple[float, float]:
    if tdf is None or len(tdf) < 2 or "entry_time" not in tdf.columns:
        return np.nan, np.nan

    mid = tdf["entry_time"].median()
    h1 = tdf[tdf["entry_time"] < mid].dropna(subset=["pnl"]).copy()
    h2 = tdf[tdf["entry_time"] >= mid].dropna(subset=["pnl"]).copy()
    return compute_profit_factor(h1), compute_profit_factor(h2)


def print_summary(tdf: pd.DataFrame, label: str = "FULL"):
    n = len(tdf)
    if n == 0:
        print(f"[{label}] No trades."); return

    print(f"{'='*60}")
    print(f"  {label} -- {RUN_VERSION}")
    print(f"{'='*60}")
    print(f"Trades                    : {n}")

    has_exits = int(tdf["exit_price"].notna().sum())
    if has_exits > 0:
        closed = tdf.dropna(subset=["pnl"]).copy()
        wins = int((closed["pnl"] > 0).sum())
        losses = int((closed["pnl"] < 0).sum())
        flat = int((closed["pnl"] == 0).sum())
        wr = wins / len(closed) * 100 if len(closed) > 0 else 0
        avg_pnl = closed["pnl"].mean()
        total_pnl = closed["pnl"].sum()
        avg_win = closed.loc[closed["pnl"] > 0, "pnl"].mean() if wins > 0 else 0
        avg_loss = closed.loc[closed["pnl"] < 0, "pnl"].mean() if losses > 0 else 0

        print(f"Win rate                  : {wr:.1f}%  ({wins}W / {losses}L / {flat}F)")
        print(f"Avg PnL per trade         : {avg_pnl:.4f}")
        print(f"Total PnL                 : {total_pnl:.4f}")
        print(f"Avg win                   : {avg_win:.4f}")
        print(f"Avg loss                  : {avg_loss:.4f}")
        pf = compute_profit_factor(closed)
        if np.isfinite(pf):
            print(f"Profit factor             : {pf:.2f}")

        closed["hold_bars"] = closed["exit_idx"] - closed["entry_idx"]
        print(f"Avg hold (15m bars)       : {closed['hold_bars'].mean():.1f}")
        print(f"Avg hold (hours)          : {closed['hold_bars'].mean() * 0.25:.1f}")

    longs = int((tdf["side"] == "long").sum())
    shorts = int((tdf["side"] == "short").sum())
    print(f"Longs / Shorts            : {longs} / {shorts}")

    if has_exits > 0:
        print("\n--- Exit Reason Breakdown ---")
        rc = tdf["exit_reason"].value_counts()
        rp = (rc / rc.sum() * 100)
        for reason in rc.index:
            cnt = rc[reason]; pct = rp[reason]
            sub = closed[closed["exit_reason"] == reason]
            sub_pnl = sub["pnl"].mean() if len(sub) > 0 else 0
            sub_wr = (sub["pnl"] > 0).mean() * 100 if len(sub) > 0 else 0
            print(f"  {reason:<20s}: {cnt:4d} ({pct:5.1f}%)  avgPnL={sub_pnl:+.4f}  WR={sub_wr:.0f}%")

        print("\n--- Target Checks ---")
        print(f"  sl_rej  : {rp.get('sl_rej', 0):.0f}%  (target: <50%)")
        print(f"  session_end: {rp.get('session_end', 0):.0f}%  (target: ~0%)")

    if "va_accept_at_entry" in tdf.columns:
        va = tdf["va_accept_at_entry"]
        print("\n--- VA Accept at Entry ---")
        print(f"  mean={va.mean():.2f}, min={va.min():.2f}, max={va.max():.2f}")


# -- Sanity --
if len(trades_df) > 0 and "exit_reason" in trades_df.columns:
    assert "sl_zone" not in set(trades_df["exit_reason"].dropna().unique()), "Stale results."
    print(f"Exit reasons: {sorted(trades_df['exit_reason'].dropna().unique())}\n")

print_summary(trades_df, "FULL DATASET")

# -- Split half --
if len(trades_df) >= 10:
    mid = trades_df["entry_time"].median()
    h1 = trades_df[trades_df["entry_time"] < mid]
    h2 = trades_df[trades_df["entry_time"] >= mid]
    print("\n"); print_summary(h1, "FIRST HALF")
    print("\n"); print_summary(h2, "SECOND HALF")



[FULL DATASET] No trades.


In [34]:
# -- Visualization: 15m candles + 1H LVN zones + trades -------------------

def plot_window_15m(
    df_15m: pd.DataFrame,
    df_1h: pd.DataFrame,
    session_states_1h: Dict[int, SessionState],
    trades_df: pd.DataFrame,
    start: int = 0,
    window: int = 400,  # 400 x 15m = ~4 days
):
    """Plot 15m candles with 1H LVN zones overlaid and trade markers."""
    end = min(start + window, len(df_15m))
    sub = df_15m.iloc[start:end].copy()
    sub["pos"] = range(len(sub))

    fig, ax = plt.subplots(figsize=(20, 7))

    # Candlesticks (15m)
    for _, row in sub.iterrows():
        p = row["pos"]
        color = "green" if row["close"] >= row["open"] else "red"
        ax.plot([p, p], [row["low"], row["high"]], color="gray", linewidth=0.3)
        body_lo = min(row["open"], row["close"])
        body_hi = max(row["open"], row["close"])
        ax.add_patch(mpatches.Rectangle(
            (p - 0.35, body_lo), 0.7, max(body_hi - body_lo, row["high"] * 1e-6),
            facecolor=color, edgecolor=color, linewidth=0.3,
        ))

    # LVN zones from 1H sessions
    if "session_id" in df_1h.columns:
        plotted_zones = set()
        for sk, state in session_states_1h.items():
            if state is None:
                continue
            for z in state.zones:
                zk = (z.session_key_created, z.start_bin, z.end_bin)
                if zk in plotted_zones:
                    continue
                plotted_zones.add(zk)
                ax.add_patch(mpatches.Rectangle(
                    (0, z.bottom), len(sub), z.top - z.bottom,
                    facecolor="blue", edgecolor="blue", alpha=0.08, linewidth=0.5,
                ))

    # Trade markers
    if len(trades_df) > 0:
        for _, t in trades_df.iterrows():
            idx = int(t["entry_idx"])
            if idx < start or idx >= end:
                continue
            p = idx - start
            color = "lime" if t["side"] == "long" else "magenta"
            marker = "^" if t["side"] == "long" else "v"
            ax.scatter(p, t["entry_price"], color=color, marker=marker,
                       s=80, zorder=5, edgecolors="black")
            if pd.notna(t.get("exit_idx")):
                ex_idx = int(t["exit_idx"])
                if start <= ex_idx < end:
                    ep = ex_idx - start
                    ex_color = {"tp_poc": "gold", "sl_rej": "red",
                                "time_stop": "gray", "session_end": "orange"}.get(
                        t.get("exit_reason", ""), "orange")
                    ax.scatter(ep, t["exit_price"], color=ex_color,
                               marker="x", s=60, zorder=5)

    n_ticks = min(20, len(sub))
    tick_step = max(1, len(sub) // n_ticks)
    tick_pos = list(range(0, len(sub), tick_step))
    tick_labels = [
        sub.index[p].strftime("%m-%d %H:%M")
        if hasattr(sub.index[p], "strftime") else str(sub.index[p])
        for p in tick_pos
    ]
    ax.set_xticks(tick_pos)
    ax.set_xticklabels(tick_labels, rotation=45, fontsize=6)
    ax.set_title(f"{RUN_VERSION}  |  15m bars {start}:{end}", fontsize=11)
    ax.set_ylabel("Price")
    ax.grid(True, alpha=0.15)
    plt.tight_layout()
    plt.show()


# Note: for v7, session_states aren't stored the same way.
# To plot, collect from 1H df if needed.
# For now, skip auto-plot; call plot_window_15m() manually.
print("Plot function plot_window_15m() ready. Call with desired window.")

Plot function plot_window_15m() ready. Call with desired window.


In [ ]:
# -- Parameter sweeps --------------------------------------------------------

def compute_metrics_for_sweep(trades: pd.DataFrame) -> Dict[str, float]:
    if trades is None or len(trades) == 0:
        return {"Trades": 0, "PF": np.nan, "sl_rej": np.nan, "Avg PnL": np.nan}

    closed = trades.dropna(subset=["pnl"]).copy()
    if len(closed) == 0:
        return {"Trades": int(len(trades)), "PF": np.nan, "sl_rej": np.nan, "Avg PnL": np.nan}

    gross_profit = closed.loc[closed["pnl"] > 0, "pnl"].sum()
    gross_loss = -closed.loc[closed["pnl"] < 0, "pnl"].sum()
    pf = (gross_profit / gross_loss) if gross_loss > 0 else np.nan
    sl_rej_pct = (closed["exit_reason"].eq("sl_rej").mean() * 100.0)

    return {
        "Trades": int(len(closed)),
        "PF": float(pf) if np.isfinite(pf) else np.nan,
        "sl_rej": float(sl_rej_pct),
        "Avg PnL": float(closed["pnl"].mean()),
    }


def run_single_sweep(cfg_overrides: dict) -> Dict[str, float]:
    cfg = dict(CONFIG)
    cfg.update(cfg_overrides)
    trades, *_ = backtest_dual_tf(df_1h, df_15m, h1_index_map, cfg)
    return compute_metrics_for_sweep(trades)


# Experiment 1 — VA filter
va_thresholds = [0.3, 0.4, 0.5, 0.6]
va_rows = []
for threshold in va_thresholds:
    metrics = run_single_sweep({
        "require_rejection": True,
        "use_va_containment_filter": True,
        "va_containment_threshold": threshold,
    })
    metrics["VA_accept >="] = threshold
    va_rows.append(metrics)

va_sweep_df = pd.DataFrame(va_rows)[["VA_accept >=", "Trades", "PF", "sl_rej", "Avg PnL"]]
print("Experiment 1 — VA filter")
display(va_sweep_df)


# Experiment 2 — rejection strength
wick_ratios = [0.3, 0.4, 0.5, 0.6]
wick_rows = []
for ratio in wick_ratios:
    metrics = run_single_sweep({
        "require_rejection": True,
        "min_rejection_wick_ratio": ratio,
    })
    metrics["min_rejection_wick_ratio"] = ratio
    wick_rows.append(metrics)

wick_sweep_df = pd.DataFrame(wick_rows)[["min_rejection_wick_ratio", "Trades", "PF", "sl_rej", "Avg PnL"]]
print("Experiment 2 — rejection strength")
display(wick_sweep_df)


### Research note — current default candidate

- VA containment threshold sweep showed improving profit factor as the threshold increased across tested levels.
- Rejection wick-ratio sweep degraded profit factor at every tested wick-ratio threshold.
- Current preferred candidate: `va_containment_threshold = 0.60`, `min_lvn_height_pct = 10.0`, `min_edge_bps = 30`, and `min_rejection_wick_ratio = 0.0` (wick filter effectively disabled).


### Research sweep — context tightening for lower `sl_rej`

This sweep is designed to reduce `sl_rej` by tightening contextual acceptance and LVN significance.
We are **not** changing TP/SL logic or the 1H/15m execution structure in this experiment.
Selection emphasis is on both strong overall PF and out-of-sample stability (second-half PF).

`min_lvn_height_pct` is interpreted as a **literal percent of total profile height (rows)**:
`min_lvn_rows = max(1, ceil((min_lvn_height_pct / 100.0) * num_rows))`.


In [ ]:
# -- Research sweep: VA containment x LVN height (% of profile rows) x minimum edge -----------------
from itertools import product

BASELINE_SWEEP_CFG = {
    "use_va_containment_filter": True,
    "va_containment_threshold": 0.60,
    "min_rejection_wick_ratio": 0.0,
    "debug_h1_trace_bars": 0,
}


def run_research_sweep(
    va_thresholds=(0.50, 0.55, 0.60, 0.65),
    lvn_heights=(5.0, 10.0, 15.0, 20.0),  # literal percent of profile height
    min_edges=(20, 25, 30, 35),
) -> pd.DataFrame:
    rows = []

    for va_thr, lvn_h, min_edge in product(va_thresholds, lvn_heights, min_edges):
        cfg = dict(CONFIG)  # fresh copy each run for deterministic behavior
        cfg.update(BASELINE_SWEEP_CFG)
        cfg.update({
            "va_containment_threshold": float(va_thr),
            "min_lvn_height_pct": float(lvn_h),
            "min_edge_bps": float(min_edge),
        })

        trades, *_ = backtest_dual_tf(df_1h, df_15m, h1_index_map, cfg)
        metrics = compute_trade_metrics(trades)
        first_pf, second_pf = compute_split_profit_factors(trades)
        computed_min_rows = compute_min_lvn_rows(float(lvn_h), int(cfg["num_rows"]))

        row = {
            "va_containment_threshold": float(va_thr),
            "min_lvn_height_pct": float(lvn_h),
            "computed_min_lvn_rows": int(computed_min_rows),
            "min_edge_bps": float(min_edge),
            **metrics,
            "First half PF": first_pf,
            "Second half PF": second_pf,
        }
        row["pf_gap"] = row["First half PF"] - row["Second half PF"]
        row["stable_oos"] = bool(np.isfinite(row["Second half PF"]) and row["Second half PF"] >= 1.0)
        rows.append(row)

    df = pd.DataFrame(rows)
    ordered_cols = [
        "va_containment_threshold",
        "min_lvn_height_pct",
        "computed_min_lvn_rows",
        "min_edge_bps",
        "Trades",
        "Win rate",
        "Profit factor",
        "Avg PnL per trade",
        "Total PnL",
        "sl_rej percentage",
        "tp_poc percentage",
        "Avg hold (hours)",
        "First half PF",
        "Second half PF",
        "pf_gap",
        "stable_oos",
    ]
    return df[ordered_cols].sort_values(["Profit factor", "Avg PnL per trade"], ascending=False).reset_index(drop=True)


research_sweep_df = run_research_sweep()
print(f"Research grid completed: {len(research_sweep_df)} runs (expected 64).")
display(research_sweep_df)

print("\nTop 15 by Profit Factor")
display(research_sweep_df.sort_values(["Profit factor", "Second half PF"], ascending=False).head(15))

print("\nTop 15 by Avg PnL per trade")
display(research_sweep_df.sort_values(["Avg PnL per trade", "Profit factor"], ascending=False).head(15))

print("\nTop 15 by Second half PF")
display(research_sweep_df.sort_values(["Second half PF", "Profit factor"], ascending=False).head(15))

print("\nTop 15 by lowest sl_rej among runs with PF > 1.5")
low_sl_high_pf = research_sweep_df[research_sweep_df["Profit factor"] > 1.5].copy()
display(low_sl_high_pf.sort_values(["sl_rej percentage", "Second half PF"], ascending=[True, False]).head(15))

print("\nStrict practical filter")
strict_practical_candidates = research_sweep_df[
    (research_sweep_df["Trades"] >= 150)
    & (research_sweep_df["Profit factor"] >= 1.5)
    & (research_sweep_df["Second half PF"] >= 1.2)
    & (research_sweep_df["sl_rej percentage"] <= 60)
].copy()

strict_practical_candidates = strict_practical_candidates.sort_values(
    ["Second half PF", "Profit factor", "Avg PnL per trade"],
    ascending=[False, False, False],
).reset_index(drop=True)

display(strict_practical_candidates)

print("\nRelaxed practical filter")
relaxed_practical_candidates = research_sweep_df[
    (research_sweep_df["Trades"] >= 150)
    & (research_sweep_df["Profit factor"] >= 1.5)
    & (research_sweep_df["Second half PF"] >= 1.2)
    & (research_sweep_df["sl_rej percentage"] <= 75)
].copy()

relaxed_practical_candidates = relaxed_practical_candidates.sort_values(
    ["Second half PF", "Profit factor", "Avg PnL per trade"],
    ascending=[False, False, False],
).reset_index(drop=True)

display(relaxed_practical_candidates)

print("\nRecommended candidate configs (top 5 strict practical candidates)")
if len(strict_practical_candidates) == 0:
    print("No strict practical candidates matched all hard filters.")
else:
    for i, row in strict_practical_candidates.head(5).iterrows():
        print(
            f"#{i+1} | va_thr={row['va_containment_threshold']:.2f} "
            f"| lvn_h={row['min_lvn_height_pct']:.2f}% | min_rows={int(row['computed_min_lvn_rows'])} "
            f"| min_edge={row['min_edge_bps']:.0f} | trades={int(row['Trades'])} "
            f"| PF={row['Profit factor']:.2f} | PF2={row['Second half PF']:.2f} "
            f"| sl_rej={row['sl_rej percentage']:.1f}% | avgPnL={row['Avg PnL per trade']:.4f}"
        )


### Comparison — current default vs previous default vs new candidate

Side-by-side comparison of the baseline configurations:
- Current default run (from `CONFIG`)
- Previous default (`va_containment_threshold = 0.55`, `min_lvn_height_pct = 15.0`, `min_edge_bps = 35`)
- New candidate (`va_containment_threshold = 0.60`, `min_lvn_height_pct = 10.0`, `min_edge_bps = 30`)


In [ ]:
# -- Baseline comparison: current default vs previous default vs new candidate --
comparison_runs = {
    "Current default (CONFIG)": {},
    "Previous default (0.55 / 15 / 35)": {
        "va_containment_threshold": 0.55,
        "min_lvn_height_pct": 15.0,
        "min_edge_bps": 35,
        "min_rejection_wick_ratio": 0.0,
    },
    "New candidate (0.60 / 10 / 30)": {
        "va_containment_threshold": 0.60,
        "min_lvn_height_pct": 10.0,
        "min_edge_bps": 30,
        "min_rejection_wick_ratio": 0.0,
    },
}

comparison_rows = []
for label, overrides in comparison_runs.items():
    cfg = dict(CONFIG)
    cfg.update(overrides)

    trades_cmp, *_ = backtest_dual_tf(df_1h, df_15m, h1_index_map, cfg)
    metrics_cmp = compute_trade_metrics(trades_cmp)
    first_pf_cmp, second_pf_cmp = compute_split_profit_factors(trades_cmp)

    comparison_rows.append({
        "Run": label,
        "Trades": metrics_cmp["Trades"],
        "PF": metrics_cmp["Profit factor"],
        "Avg PnL": metrics_cmp["Avg PnL per trade"],
        "Total PnL": metrics_cmp["Total PnL"],
        "sl_rej": metrics_cmp["sl_rej percentage"],
        "First half PF": first_pf_cmp,
        "Second half PF": second_pf_cmp,
    })

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)
